# 02 - LangGraph Workflow：建立多步驟 LLM 工作流

本筆記本示範如何直接使用 LangGraph `StateGraph` API 建構 workflow，
並透過 `call_llm` node 搭配 MLflow span 追蹤 LLM 呼叫細節。

## 核心概念

本框架直接採用 LangGraph 原生架構，不再包裝額外抽象層：

- **`state.py`**：定義 `TypedDict` state schemas（`BaseState`、`LLMState`）
- **`nodes.py`**：提供通用 node functions（如 `create_call_llm_node`）
- **`build_workflow.py`**：存放預建的 workflow 範例

你可以直接使用 LangGraph 的 `StateGraph`、`add_node`、`add_edge`、`set_entry_point` 等 API。

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from app.utils.config import init_config
from app.tracking import init_mlflow
from app.workflow import LLMState, create_llm_state, create_call_llm_node
from langgraph.graph import END, StateGraph

cfg = init_config()
init_mlflow(cfg)
print("設定與 MLflow 初始化完成。")

## LLMState 結構

| 欄位 | 型別 | 說明 |
|------|------|------|
| `messages` | `list` | 對話訊息（`add_messages` reducer 自動合併） |
| `metadata` | `dict` | 任意 key-value 中繼資料 |
| `llm_response` | `str` | LLM 回應內容 |
| `token_usage` | `dict` | Token 使用量 |
| `model` | `str` | 使用的模型名稱 |
| `error` | `str \| None` | 錯誤訊息 |

## 範例一：使用預建 workflow

`build_simple_chain()` 建構最簡單的 LLM chain：user message → call_llm → END

In [ ]:
from app.workflow.build_workflow import build_simple_chain
from llm_service import LLMClient

# 建立 LLM client
client = LLMClient(config={
    "base_url": cfg.model.base_url,
    "model": cfg.model.name,
})

# 使用預建的 simple chain
graph = build_simple_chain(client, system_prompt="你是一個有用的中文助手。")

# 建立初始狀態並執行
initial = create_llm_state(
    messages=[{"role": "user", "content": "什麼是 LangGraph？"}],
)

try:
    result = graph.invoke(initial)
    print("=== Workflow 結果 ===")
    print(f"LLM 回應: {result['llm_response'][:200]}")
    print(f"Token 用量: {result['token_usage']}")
    print(f"模型: {result['model']}")
except Exception as e:
    print(f"[警告] 執行失敗：{e}")
    print("請確認 LLM 服務已啟動。")

## 範例二：自定義多步驟 Workflow

直接使用 LangGraph `StateGraph` API 建構自定義 workflow：

```
preprocess → call_llm → postprocess → END
```

In [ ]:
# 定義自定義 node functions
def preprocess(state: dict) -> dict:
    """前處理：擷取 user message 並加入分析指令。"""
    messages = state.get("messages", [])
    user_text = ""
    for msg in reversed(messages):
        if isinstance(msg, dict) and msg.get("role") == "user":
            user_text = msg["content"]
            break

    print(f"[preprocess] 擷取輸入: {user_text[:50]}...")

    # 將原始 user message 修改為帶指令的版本
    enhanced_prompt = f"請針對以下內容進行分析並提供摘要：\n\n{user_text}"
    return {
        "messages": [{"role": "user", "content": enhanced_prompt}],
        "metadata": {**state.get("metadata", {}), "preprocessed": True},
    }


def postprocess(state: dict) -> dict:
    """後處理：清理 LLM 回應，計算統計。"""
    response = state.get("llm_response", "")
    token_usage = state.get("token_usage", {})

    stats = {
        "response_length": len(response),
        "total_tokens": token_usage.get("total_tokens", 0),
    }

    print(f"[postprocess] 回應長度: {stats['response_length']} 字元, Tokens: {stats['total_tokens']}")

    return {
        "metadata": {**state.get("metadata", {}), "stats": stats},
    }


print("自定義 node 定義完成。")

In [ ]:
# 使用 LangGraph StateGraph 組裝 workflow
call_llm = create_call_llm_node(client, default_system_prompt="你是專業的文件分析助手。")

graph = StateGraph(LLMState)
graph.add_node("preprocess", preprocess)
graph.add_node("call_llm", call_llm)
graph.add_node("postprocess", postprocess)

graph.set_entry_point("preprocess")
graph.add_edge("preprocess", "call_llm")
graph.add_edge("call_llm", "postprocess")
graph.add_edge("postprocess", END)

compiled = graph.compile()
print("Workflow 組裝完成：preprocess → call_llm → postprocess → END")

In [ ]:
# 執行自定義 workflow
initial = create_llm_state(
    messages=[{"role": "user", "content": "LLM 是一種大型語言模型，能夠理解並生成自然語言。它透過大量文本訓練，學會了語言的統計規律。"}],
)

try:
    final = compiled.invoke(initial)
    print("\n=== Workflow 執行完成 ===")
    print(f"LLM 回應（前 200 字）: {final['llm_response'][:200]}")
    print(f"\n訊息記錄（共 {len(final['messages'])} 則）")
    print(f"Metadata: {final['metadata']}")
    print(f"Error: {final['error']}")
except Exception as e:
    print(f"[警告] Workflow 執行失敗：{e}")

## 範例三：帶條件分支的 Workflow

使用 `add_conditional_edges` 根據 state 動態決定下一個節點。

In [ ]:
def router(state: dict) -> str:
    """根據 metadata 中的 task 決定路由。"""
    task = state.get("metadata", {}).get("task", "default")
    if task == "summarize":
        return "summarize"
    return "analyze"


def summarize_node(state: dict) -> dict:
    print("[summarize] 執行摘要任務")
    return {"llm_response": "（摘要模式）" + state.get("llm_response", "")}


def analyze_node(state: dict) -> dict:
    print("[analyze] 執行分析任務")
    return {"llm_response": "（分析模式）" + state.get("llm_response", "")}


# 建構帶條件分支的 workflow
call_llm = create_call_llm_node(client)

g = StateGraph(LLMState)
g.add_node("call_llm", call_llm)
g.add_node("summarize", summarize_node)
g.add_node("analyze", analyze_node)

g.set_entry_point("call_llm")
g.add_conditional_edges("call_llm", router, {"summarize": "summarize", "analyze": "analyze"})
g.add_edge("summarize", END)
g.add_edge("analyze", END)

conditional_graph = g.compile()

# 測試不同路由
for task in ["summarize", "analyze"]:
    state = create_llm_state(
        messages=[{"role": "user", "content": "Hello"}],
        metadata={"task": task},
    )
    try:
        result = conditional_graph.invoke(state)
        print(f"  Task={task} → 回應前綴: {result['llm_response'][:30]}")
    except Exception as e:
        print(f"  Task={task} → 失敗: {e}")

## 小結

| 元件 | 說明 |
|------|------|
| `LLMState` | Workflow 共享狀態（messages, metadata, llm_response, token_usage...） |
| `create_call_llm_node(client)` | 建立帶 MLflow span 追蹤的 LLM 呼叫 node |
| `StateGraph` | LangGraph 原生 API，直接建構 workflow |
| `build_simple_chain()` | 預建的簡單 LLM chain |
| `add_conditional_edges()` | 根據 state 動態路由 |

### MLflow Tracing

- `mlflow.langchain.autolog()` 自動追蹤所有 LangGraph 呼叫
- `call_llm` node 內部使用 `mlflow.start_span()` 記錄 reasoning、token usage
- 啟動 `mlflow ui --port 5000` 查看完整 trace

### 後續步驟

- `03_evaluation.ipynb`：使用 MLflow GenAI evaluate 進行評估
- `04_prompt_management.ipynb`：Prompt 註冊、版本管理與自動優化